L2-regularized logistic regression for binary or multiclass classification; trains a model (on `train.txt`), optimizes L2 regularization strength on `dev.txt`, and evaluates performance on `test.txt`.  Reports test accuracy with 95% confidence intervals and prints out the strongest coefficients for each class.

In [134]:
from scipy import sparse
from sklearn import linear_model
from collections import Counter
import numpy as np
import operator
import nltk
import math
import string
from scipy.stats import norm
#let's try adding TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer

In [135]:
!python -m nltk.downloader punkt

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [136]:
def load_data(filename):
    X = []
    Y = []
    with open(filename, encoding="utf-8") as file:
        for line in file:
            cols = line.strip().split("\t")
            if len(cols) < 3:
                continue
            if cols[0] == "id" and cols[1] == "label":
                continue

            idd = cols[0]
            label = cols[1].strip()
            text = cols[2]

            X.append(text)
            Y.append(label)

    return X, Y


In [137]:
class Classifier:

    def __init__(self, feature_method, trainX, trainY, devX, devY, testX, testY):
        self.feature_vocab = {}
        self.feature_method = feature_method
        self.min_feature_count=10
        self.log_reg = None

        self.trainY=trainY
        self.devY=devY
        self.testY=testY

        self.trainX = self.process(trainX, training=True)
        self.devX = self.process(devX, training=False)
        self.testX = self.process(testX, training=False)

    # Featurize entire dataset
    def featurize(self, data):
        featurized_data = []
        for text in data:
            feats = self.feature_method(text)
            featurized_data.append(feats)
        return featurized_data

    # Read dataset and returned featurized representation as sparse matrix + label array
    def process(self, X_data, training = False):

        data = self.featurize(X_data)

        if training:
            fid = 0
            feature_doc_count = Counter()
            for feats in data:
                for feat in feats:
                    feature_doc_count[feat]+= 1

            for feat in feature_doc_count:
                if feature_doc_count[feat] >= self.min_feature_count:
                    self.feature_vocab[feat] = fid
                    fid += 1

        F = len(self.feature_vocab)
        D = len(data)
        X = sparse.dok_matrix((D, F))
        for idx, feats in enumerate(data):
            for feat in feats:
                if feat in self.feature_vocab:
                    X[idx, self.feature_vocab[feat]] = feats[feat]

        return X

    # Train model and evaluate on held-out data
    def train(self):
        (D,F) = self.trainX.shape
        best_dev_accuracy=0
        best_model=None
        for C in [0.1, 1, 10, 100]:
            self.log_reg = linear_model.LogisticRegression(C = C, max_iter=1000)
            self.log_reg.fit(self.trainX, self.trainY)
            training_accuracy = self.log_reg.score(self.trainX, self.trainY)
            development_accuracy = self.log_reg.score(self.devX, self.devY)
            if development_accuracy > best_dev_accuracy:
                best_dev_accuracy=development_accuracy
                best_model=self.log_reg

#             print("C: %s, Train accuracy: %.3f, Dev accuracy: %.3f" % (C, training_accuracy, development_accuracy))

        self.log_reg=best_model


    def test(self):
        return self.log_reg.score(self.testX, self.testY)


    def printWeights(self, n=10):

        reverse_vocab=[None]*len(self.log_reg.coef_[0])
        for k in self.feature_vocab:
            reverse_vocab[self.feature_vocab[k]]=k

        # binary
        if len(self.log_reg.classes_) == 2:
              weights=self.log_reg.coef_[0]

              cat=self.log_reg.classes_[1]
              for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1))))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()

              cat=self.log_reg.classes_[0]
              for feature, weight in list(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1)))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()

        # multiclass
        else:
          for i, cat in enumerate(self.log_reg.classes_):

              weights=self.log_reg.coef_[i]

              for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1))))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()



In [138]:
import nltk
from nltk.util import ngrams
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter
import string
nltk.download('vader_lexicon')
VADER = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [139]:
import re
#code referenced from https://www.nltk.org/api/nltk.sentiment.vader.html

def binary_bow_featurize(text):
    feats = {}
    raw_text = text
    words = nltk.word_tokenize(text)
    #remove ',', ';' and '.' within the tokens
    clean_words = [word.strip(',;.') for word in words]

    for word in clean_words:
      #word=word.lower() #got better accuracy commenting this out 43 vs 37
      feats[word]= 1

    #returns a dict with 'pos', 'neg', 'neu', and 'compound' scores using VADER
    sentiment = VADER.polarity_scores(raw_text)
    feats["sentiment_pos"] = sentiment['pos']
    feats["sentiment_neg"] = sentiment['neg']
    feats["sentiment_compound"] = sentiment['compound']
    return feats

In [140]:
def confidence_intervals(accuracy, n, significance_level):
    critical_value=(1-significance_level)/2
    z_alpha=-1*norm.ppf(critical_value)
    se=math.sqrt((accuracy*(1-accuracy))/n)
    return accuracy-(se*z_alpha), accuracy+(se*z_alpha)

In [141]:
def run(trainingFile, devFile, testFile):
    trainX, trainY=load_data(trainingFile)
    devX, devY=load_data(devFile)
    testX, testY=load_data(testFile)

    simple_classifier = Classifier(binary_bow_featurize, trainX, trainY, devX, devY, testX, testY)
    simple_classifier.train()
    accuracy=simple_classifier.test()

    lower, upper=confidence_intervals(accuracy, len(testY), .95)
    print("Test accuracy for best dev model: %.3f, 95%% CIs: [%.3f %.3f]\n" % (accuracy, lower, upper))

    simple_classifier.printWeights()


In [142]:
trainingFile = '/content/sample_data/train.txt'
devFile = '/content/sample_data/dev.txt'
testFile = '/content/sample_data/test.txt'
run(trainingFile, devFile, testFile)

Test accuracy for best dev model: 0.430, 95% CIs: [0.333 0.527]

1	0.182	'
1	0.145	opportunity
1	0.139	times
1	0.138	short
1	0.132	learning
1	0.131	think
1	0.131	same
1	0.125	out
1	0.124	without
1	0.120	other

2	0.376	I
2	0.235	good
2	0.216	will
2	0.208	better
2	0.198	every
2	0.166	point
2	0.162	has
2	0.151	which
2	0.148	pay
2	0.144	who

3	0.249	its
3	0.204	Moreover
3	0.186	students
3	0.180	amount
3	0.178	between
3	0.177	those
3	0.177	true
3	0.176	course
3	0.168	'
3	0.165	much

4	0.307	provide
4	0.277	because
4	0.273	First
4	0.258	On
4	0.252	by
4	0.220	job
4	0.219	friends
4	0.206	living
4	0.201	another
4	0.198	up

5	0.507	example
5	0.334	For
5	0.319	such
5	0.294	are
5	0.241	work
5	0.230	used
5	0.226	when
5	0.212	what
5	0.211	I
5	0.196	working

6	0.199	countries
6	0.167	time
6	0.154	waste
6	0.152	at
6	0.144	sports
6	0.137	free
6	0.131	provides
6	0.128	important
6	0.127	In
6	0.125	home

7	0.188	we
7	0.175	In
7	0.167	themselves
7	0.163	skills
7	0.150	most
7	0.149	might
7	0.149	which
7	0.1